# Phase A — PCA Structural Analysis of Query Embeddings

Diagnostic: do entity-class query embeddings have a dominant principal direction (PC1) that is meaningfully distinct from the class centroid?

**Gating criterion:**
- GREENLIGHT Phase B if median entity PC1 variance ≥ 0.40 AND median |PC1-centroid cos| ≤ 0.85
- RED-LIGHT if median entity PC1 variance < 0.30 OR median |PC1-centroid cos| > 0.95
- Otherwise: ambiguous, report to user.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

SCRATCH  = Path('/home/ishana/scratch')
DATA     = SCRATCH / 'data' / 'classes'
PROJ     = Path('/home/ishana/projects/llm_jam_universal')
RESULTS  = PROJ / 'results'
FIGS     = SCRATCH / 'results'

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

df = pd.read_csv(RESULTS / 'pca_analysis.csv')
para_df   = df[df.class_type == 'paraphrase']
entity_df = df[df.class_type == 'entity']

print(f"Paraphrase classes: {len(para_df)}")
print(f"Entity classes:     {len(entity_df)}")
print(df.dtypes)


## 1. Summary Statistics

In [ ]:
print("=== PC1 Explained Variance ===")
for ct, sub in [('paraphrase', para_df), ('entity', entity_df)]:
    v = sub['pc1_explained_var'].values
    print(f"  {ct}: mean={v.mean():.3f}  median={np.median(v):.3f}  "
          f"min={v.min():.3f}  max={v.max():.3f}")

print()
print("=== |PC1-centroid cosine| ===")
for ct, sub in [('paraphrase', para_df), ('entity', entity_df)]:
    v = np.abs(sub['pc1_centroid_cosine'].values)
    print(f"  {ct}: mean={v.mean():.3f}  median={np.median(v):.3f}  "
          f"min={v.min():.3f}  max={v.max():.3f}")

print()
print("=== Intrinsic dim (90% var) ===")
for ct, sub in [('paraphrase', para_df), ('entity', entity_df)]:
    v = sub['intrinsic_dim_90'].values
    print(f"  {ct}: mean={v.mean():.2f}  median={np.median(v):.1f}  "
          f"min={v.min():.0f}  max={v.max():.0f}")

# Gating
med_pc1   = float(np.median(entity_df['pc1_explained_var']))
med_cos   = float(np.median(np.abs(entity_df['pc1_centroid_cosine'])))
print(f"\nGating: entity median PC1={med_pc1:.3f}, |cos|={med_cos:.3f}")
if med_pc1 >= 0.40 and med_cos <= 0.85:
    print("  → GREENLIGHT")
elif med_pc1 < 0.30 or med_cos > 0.95:
    print("  → RED-LIGHT")
else:
    print("  → AMBIGUOUS")


## 2. Main Figure: Explained-Variance Histograms

Paraphrase vs Entity classes, side by side. Candidate paper figure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

COLORS = {'paraphrase': '#1f77b4', 'entity': '#d62728'}

# ─ PC1 explained variance ─────────────────────────────────────────────────────
ax = axes[0]
bins = np.linspace(0, 1, 21)
for ct, sub in [('paraphrase', para_df), ('entity', entity_df)]:
    v   = sub['pc1_explained_var'].values
    med = np.median(v)
    ax.hist(v, bins=bins, color=COLORS[ct], alpha=0.6, label=f'{ct} (n={len(sub)})', density=False)
    ax.axvline(med, color=COLORS[ct], lw=2, ls='--', label=f'{ct} median={med:.2f}')
ax.set_xlabel('PC1 explained variance')
ax.set_ylabel('Count')
ax.set_title('PC1 Explained Variance')
ax.legend(fontsize=8)

# ─ Intrinsic dimensionality ───────────────────────────────────────────────────
ax = axes[1]
max_dim = max(df['intrinsic_dim_90'].max(), 5)
bins_id = np.arange(0.5, max_dim + 1.5)
width   = 0.35
for offset, (ct, sub) in zip([-width/2, width/2],
                               [('paraphrase', para_df), ('entity', entity_df)]):
    v     = sub['intrinsic_dim_90'].values
    vals, cnts = np.unique(v, return_counts=True)
    ax.bar(vals + offset, cnts, width=width, color=COLORS[ct],
           alpha=0.7, label=f'{ct} (med={np.median(v):.0f})')
ax.set_xlabel('Intrinsic dim (PCs for ≥90% var)')
ax.set_ylabel('Count')
ax.set_title('Intrinsic Dimensionality')
ax.legend(fontsize=8)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

# ─ |PC1-centroid cosine| ──────────────────────────────────────────────────────
ax = axes[2]
bins_c = np.linspace(0, 1, 21)
for ct, sub in [('paraphrase', para_df), ('entity', entity_df)]:
    v   = np.abs(sub['pc1_centroid_cosine'].values)
    med = np.median(v)
    ax.hist(v, bins=bins_c, color=COLORS[ct], alpha=0.6, label=f'{ct}', density=False)
    ax.axvline(med, color=COLORS[ct], lw=2, ls='--', label=f'{ct} median={med:.2f}')
ax.set_xlabel('|PC1 · centroid| cosine')
ax.set_ylabel('Count')
ax.set_title('PC1 vs Centroid Alignment\n(high = PC1 redundant with centroid)')
ax.legend(fontsize=8)

plt.suptitle('Phase A — PCA Structural Analysis of Query Classes', fontsize=13)
plt.tight_layout()
fig.savefig(FIGS / 'pca_analysis_histograms.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved pca_analysis_histograms.png")


## 3. Per-Class Breakdown (Entity)

In [ ]:
with open(DATA / 'entity_classes.json') as f:
    entity_json = json.load(f)
eid_to_name = {c.get('class_id', f'entity_{i:02d}'): c['primary_entity']
               for i, c in enumerate(entity_json)}

edf = entity_df.copy()
edf['entity'] = edf['class_id'].map(eid_to_name)
edf = edf.sort_values('pc1_explained_var', ascending=False)
display_cols = ['class_id', 'entity', 'n_queries', 'within_class_sim',
                'pc1_explained_var', 'pc2_explained_var',
                'pc1_centroid_cosine', 'intrinsic_dim_90']
print(edf[display_cols].to_string(index=False, float_format='{:.3f}'.format))


## 4. Scatter: Within-Class Sim vs PC1 Variance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, (y_col, y_lbl) in zip(axes, [
    ('pc1_explained_var',   'PC1 explained variance'),
    ('pc1_centroid_cosine', 'PC1-centroid cosine'),
]):
    for ct, sub in [('paraphrase', para_df), ('entity', entity_df)]:
        ax.scatter(sub['within_class_sim'], sub[y_col],
                   color=COLORS[ct], alpha=0.6, label=ct, s=40)
    ax.set_xlabel('Within-class cosine similarity')
    ax.set_ylabel(y_lbl)
    ax.set_title(y_lbl + ' vs within-class sim')
    ax.legend(fontsize=9)

plt.suptitle('Phase A — PCA vs Within-Class Similarity', fontsize=13)
plt.tight_layout()
fig.savefig(FIGS / 'pca_vs_sim_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Gating Verdict

In [ ]:
med_pc1 = float(np.median(entity_df['pc1_explained_var']))
med_cos = float(np.median(np.abs(entity_df['pc1_centroid_cosine'])))

print(f"Entity median PC1 explained variance : {med_pc1:.3f}")
print(f"Entity median |PC1-centroid cosine|  : {med_cos:.3f}")
print()
if med_pc1 >= 0.40 and med_cos <= 0.85:
    verdict = "GREENLIGHT"
    reason  = "PC1 is strong and distinct from centroid — Phase B is worth running."
elif med_pc1 < 0.30 or med_cos > 0.95:
    verdict = "RED-LIGHT"
    reason  = ("PC1 is weak or redundant with centroid — Phase B unlikely to help. "
               "Phase A alone is the paper-relevant result.")
else:
    verdict = "AMBIGUOUS"
    reason  = "In between the thresholds — consult user before proceeding."

print(f"PHASE A VERDICT: {verdict}")
print(reason)
